In [1]:
import os
import json
import time
import logging
from pathlib import Path
from datetime import datetime, timezone

In [2]:
BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
LOG_DIR = BASE_DIR / "logs"

for directory in [DATA_DIR, OUTPUT_DIR, LOG_DIR]:
    directory.mkdir(exist_ok=True)

print(f"Project directory: {BASE_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Log directory: {LOG_DIR}")

Project directory: c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice
Data directory: c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\data
Output directory: c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\outputs
Log directory: c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\logs


In [3]:
DATA_SOURCES = [
    {
        "title": "Attention Is All You Need",
        "url": "https://arxiv.org/pdf/1706.03762"
    },
    {
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "url": "https://arxiv.org/pdf/2005.11401"
    }
]

DATA_SOURCES

[{'title': 'Attention Is All You Need',
  'url': 'https://arxiv.org/pdf/1706.03762'},
 {'title': 'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks',
  'url': 'https://arxiv.org/pdf/2005.11401'}]

In [4]:
import requests

for source in DATA_SOURCES:
    response = requests.get(source["url"], timeout=30)

    if response.status_code == 200:
        filename = source["title"].replace(" ", "_") + ".pdf"
        file_path = DATA_DIR / filename

        with open(file_path, "wb") as f:
            f.write(response.content)

        print(f"Downloaded: {filename}")
    else:
        print(f"Failed: {source['title']} | Status: {response.status_code}")

Downloaded: Attention_Is_All_You_Need.pdf
Downloaded: Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.pdf


In [5]:
files = list(DATA_DIR.iterdir())

print(f"Files collected: {len(files)}")

for file in files:
    print(f"- {file.name} | {file.stat().st_size / 1024:.2f} KB")

Files collected: 2
- Attention_Is_All_You_Need.pdf | 2163.32 KB
- Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.pdf | 864.57 KB


In [6]:
for file in DATA_DIR.iterdir():
    print(
        f"Name: {file.name}\n"
        f"Type: {file.suffix}\n"
        f"Size: {file.stat().st_size / 1024:.2f} KB\n"
        f"{'-' * 50}"
    )

Name: Attention_Is_All_You_Need.pdf
Type: .pdf
Size: 2163.32 KB
--------------------------------------------------
Name: Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.pdf
Type: .pdf
Size: 864.57 KB
--------------------------------------------------


In [ ]:
import pymupdf
from docx import Document
from openpyxl import load_workbook

for file in DATA_DIR.glob("*.pdf"):
    doc = pymupdf.open(file)

    print(f"\n{file.name}")
    print(f"Pages: {len(doc)}")

    for i in range(min(3, len(doc))):
        text = doc[i].get_text()

        print(f"\n--- Page {i + 1} ---")
        print(text[:500])


Attention_Is_All_You_Need.pdf
Pages: 15

--- Page 1 ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K

--- Page 2 ---
1
Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24,

In [10]:
for file in DATA_DIR.glob("*.pdf"):
    doc = pymupdf.open(file)

    total_chars = 0
    total_words = 0

    for page in doc:
        text = page.get_text()
        total_chars += len(text)
        total_words += len(text.split())

    print(
        f"{file.name}\n"
        f"Pages: {len(doc)}\n"
        f"Characters: {total_chars:,}\n"
        f"Words: {total_words:,}\n"
        f"{'-' * 50}"
    )

Attention_Is_All_You_Need.pdf
Pages: 15
Characters: 39,498
Words: 6,095
--------------------------------------------------
Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.pdf
Pages: 19
Characters: 69,078
Words: 9,886
--------------------------------------------------


In [ ]:
from datasets import load_dataset
import pandas as pd

# 1. Load the dataset from Hugging Face (e.g., 'rotten_tomatoes' or any dataset ID)
dataset = load_dataset("rotten_tomatoes", split="train")

# 2. Convert the Hugging Face dataset split to a Pandas DataFrame
df = dataset.to_pandas()

# 3. Export the DataFrame to an Excel file
df.to_excel("huggingface_dataset.xlsx", index=False)

print("Dataset successfully saved as huggingface_dataset.xlsx!")


In [11]:
documents = []

for file in DATA_DIR.glob("*.pdf"):
    doc = pymupdf.open(file)

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text()

        documents.append({
            "document": file.name,
            "page": page_number,
            "text": text
        })

print(f"Total page records: {len(documents)}")

Total page records: 34


In [12]:
documents[0]

{'document': 'Attention_Is_All_You_Need.pdf',
 'page': 1,
 'text': 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer

In [13]:
empty_pages = [
    doc for doc in documents
    if not doc["text"].strip()
]

print(f"Total pages: {len(documents)}")
print(f"Empty pages: {len(empty_pages)}")
print(f"Non-empty pages: {len(documents) - len(empty_pages)}")

Total pages: 34
Empty pages: 0
Non-empty pages: 34


In [15]:
for doc in documents[:3]:
    print(f"\n{'=' * 80}")
    print(f"Document: {doc['document']}")
    print(f"Page: {doc['page']}")
    print(f"{'=' * 80}")

    print(repr(doc["text"][:1500])) #Using repr() is intentional. It lets us see things like: \n\t,multiple spaces, broken line breaks







Document: Attention_Is_All_You_Need.pdf
Page: 1
'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on

In [16]:
for doc in documents:
    text = doc["text"]

    doc["char_count"] = len(text)
    doc["word_count"] = len(text.split())
    doc["line_count"] = len(text.splitlines())

print(documents[0])

{'document': 'Attention_Is_All_You_Need.pdf', 'page': 1, 'text': 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\

In [19]:
from docx import Document
from openpyxl import load_workbook

In [20]:
def extract_pdf(file_path):
    documents = []

    doc = pymupdf.open(file_path)

    for page_number, page in enumerate(doc, start=1):
        documents.append({
            "document": file_path.name,
            "source_type": "pdf",
            "location": f"page_{page_number}",
            "text": page.get_text(),
            "metadata": {}
        })

    return documents

In [21]:
def extract_docx(file_path):
    documents = []

    doc = Document(file_path)

    for index, paragraph in enumerate(doc.paragraphs, start=1):
        text = paragraph.text.strip()

        if text:
            documents.append({
                "document": file_path.name,
                "source_type": "docx",
                "location": f"paragraph_{index}",
                "text": text,
                "metadata": {}
            })

    return documents

In [22]:
def extract_xlsx(file_path):
    documents = []

    workbook = load_workbook(
        file_path,
        read_only=True,
        data_only=True
    )

    for sheet in workbook.worksheets:

        for row_number, row in enumerate(
            sheet.iter_rows(values_only=True),
            start=1
        ):
            values = [
                str(value)
                for value in row
                if value is not None
            ]

            if not values:
                continue

            text = " | ".join(values)

            documents.append({
                "document": file_path.name,
                "source_type": "xlsx",
                "location": f"{sheet.title}!row_{row_number}",
                "text": text,
                "metadata": {
                    "sheet": sheet.title,
                    "row": row_number
                }
            })

    return documents

In [23]:
def ingest_file(file_path):

    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        return extract_pdf(file_path)

    elif suffix == ".docx":
        return extract_docx(file_path)

    elif suffix == ".xlsx":
        return extract_xlsx(file_path)

    else:
        raise ValueError(
            f"Unsupported file format: {suffix}"
        )

In [ ]:
documents = []

for file_path in DATA_DIR.iterdir():

    if file_path.suffix.lower() in [".pdf", ".docx", ".xlsx"]:

        extracted = ingest_file(file_path)

        documents.extend(extracted)

        print(
            f"{file_path.name}: "
            f"{len(extracted)} records"
        )

print(f"\nTotal records: {len(documents)}")

In [17]:
short_pages = [
    doc for doc in documents
    if doc["word_count"] < 20
]

print(f"Suspiciously short pages: {len(short_pages)}")

for doc in short_pages[:10]:
    print(
        doc["document"],
        "| Page:", doc["page"],
        "| Words:", doc["word_count"]
    )

Suspiciously short pages: 0


In [18]:
import re


def clean_text(text: str) -> str:
    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove excessive whitespace
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text